# – Part 2: Data Preprocessing


## Step 0 – Install Required Libraries

In [1]:
! gdown --id 1v00MuC_DxJx2Jsh785qEcu1mi7GTdD31

'gdown' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
!pip install camel-tools pyarabic nltk

                                              0.0/124.7 kB ? eta -:--:--
     ---------                               30.7/124.7 kB 1.3 MB/s eta 0:00:01
     ---------------                       51.2/124.7 kB 518.5 kB/s eta 0:00:01
     ---------------------------           92.2/124.7 kB 871.5 kB/s eta 0:00:01
     ------------------------------------ 124.7/124.7 kB 665.6 kB/s eta 0:00:00
                                              0.0/126.4 kB ? eta -:--:--
     -------------------------------------- 126.4/126.4 kB 2.5 MB/s eta 0:00:00
  Using cached docopt-0.6.2-py2.py3-none-any.whl
                                              0.0/9.4 MB ? eta -:--:--
     -                                        0.2/9.4 MB 7.4 MB/s eta 0:00:02
     -                                        0.4/9.4 MB 4.0 MB/s eta 0:00:03
     ---                                      0.8/9.4 MB 6.1 MB/s eta 0:00:02
     ----                                     1.0/9.4 MB 5.2 MB/s eta 0:00:02
     -----            

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2023.3.0 requires fsspec==2023.3.0, but you have fsspec 2026.4.0 which is incompatible.


In [18]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
from camel_tools.utils.normalize import normalize_alef_ar, normalize_alef_maksura_ar
from camel_tools.tokenizers.word import simple_word_tokenize
from pyarabic.araby import strip_tashkeel
import pyarabic.araby as araby
nltk.download('stopwords')
print('All libraries installed successfully!')

All libraries installed successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kinda\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


---
## Step 1a – Load and Explore the Dataset

In [20]:
FILE_PATH = r'C:\Users\kinda\Downloads\final_dataset_text_label.csv'

df = pd.read_csv(FILE_PATH)
print('Shape:', df.shape)
df.head()

Shape: (1495, 2)


,text,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,human
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,human
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,human
3,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,human
4,الأمن السيبراني هو أحد أهم المجالات في العصر ا...,human


In [21]:
# Basic exploration
print('=== Column names ===')
print(df.columns.tolist())

print('\n=== Label distribution ===')
print(df['label'].value_counts())

print('\n=== Missing values ===')
print(df.isnull().sum())


=== Column names ===
['text', 'label']

=== Label distribution ===
label
ai                     678
human                  660
ai generated            40
ai-generated            20
humen                   20
collected               15
human-written           10
ai_generation           10
ai_generated            10
hand written            10
human_written            5
human(collected)         5
human(hand_written)      5
collect                  5
label                    1
al                       1
Name: count, dtype: int64

=== Missing values ===
text     0
label    0
dtype: int64


---
## Step 1b – Label Normalization


In [22]:
#  Label Standardization

LABEL_MAP = {
    'ai'              : 'ai_generated',
    'ai generated'    : 'ai_generated',
    'ai-generated'    : 'ai_generated',
    'ai_generation'   : 'ai_generated',
    'ai_generated'    : 'ai_generated',
    'al'              : 'ai_generated',

    'human'               : 'human_written',
    'humen'               : 'human_written',
    'human-written'       : 'human_written',
    'human_written'       : 'human_written',
    'hand written'        : 'human_written',
    'human written'       : 'human_written',
    'human(hand_written)' : 'human_written',
    'human(collected)'    : 'human_written',
    'collect'             : 'human_written',
    'collected'           : 'human_written',
}

df['label'] = df['label'].astype(str).str.strip().str.lower()

before = len(df)
df = df[df['label'] != 'label'].reset_index(drop=True)
print(f'Removed {before - len(df)} row(s) with label="label"')

df['label'] = df['label'].map(LABEL_MAP)

unmapped = df['label'].isna().sum()
if unmapped > 0:
    print(f'Warning: {unmapped} row(s) had unrecognised labels and were dropped.')
    df = df.dropna(subset=['label']).reset_index(drop=True)

print('\n=== Clean Label Distribution ===')
print(df['label'].value_counts())
print(f'\nTotal samples after label cleaning: {len(df)}')


Removed 1 row(s) with label="label"

=== Clean Label Distribution ===
label
ai_generated     759
human_written    735
Name: count, dtype: int64

Total samples after label cleaning: 1494


---
## Step 2 – Text Cleaning



In [23]:
ARABIC_STOPWORDS = set(stopwords.words('arabic'))

EXTRA_STOPWORDS = {'هذا', 'هذه', 'ذلك', 'تلك', 'هناك', 'حيث', 'كما', 'أيضا',
                   'بعض', 'كل', 'جميع', 'أي', 'لكن', 'إلا', 'لأن', 'بسبب'}
ARABIC_STOPWORDS.update(EXTRA_STOPWORDS)

def clean_arabic_text(text: str) -> str:
    if not isinstance(text, str):
        return ''

    # Remove diacritics (tashkeel)
    text = araby.strip_tashkeel(text)

    # Remove tatweel (kashida elongation)
    text = araby.strip_tatweel(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove dates (DD/MM/YYYY, YYYY-MM-DD, etc.)
    text = re.sub(r'\b(\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4})\b', ' ', text)

    # Remove Arabic-Indic digit dates
    text = re.sub(
        r'[\u0660-\u0669]{1,4}[-/.]([\u0660-\u0669]{1,2})[-/.]([\u0660-\u0669]{1,4})',
        ' ', text)

    # Remove punctuation and ALL non-Arabic characters
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)

    # Remove remaining standalone digits (Arabic-Indic / Western)
    text = re.sub(r'[\u0660-\u0669\u06F0-\u06F90-9]+', ' ', text)

    # Remove Arabic punctuation marks (،  ؛  ؟  ۔  ٪)
    text = re.sub(r'[\u060C\u061B\u061F\u06D4\u066A\u066B\u066C]', ' ', text)

    # Remove extra whitespace / newlines
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stop words (optional, we will try both ways)
    #tokens = text.split()
    #tokens = [tok for tok in tokens if tok not in ARABIC_STOPWORDS]
    #text = ' '.join(tokens)

    return text


sample_raw = df['text'].iloc[9]
sample_clean = clean_arabic_text(sample_raw)
print('\n[Before]', sample_raw[:200])
print('\n[After] ', sample_clean[:200])



[Before] ظهر الأمن السيراني مع نهاية الحرب الباردة، وظهور مصطلح حرب الإنترنت أو الحرب السيبرانية، التي جاءت مع بداية اعتماد الدول على أجهزة الكمبيوتر في مؤسساتها وتطوير وحدة المعالجة المركزية في هذه الأجهزة، ا

[After]  ظهر الأمن السيراني مع نهاية الحرب الباردة وظهور مصطلح حرب الإنترنت أو الحرب السيبرانية التي جاءت مع بداية اعتماد الدول على أجهزة الكمبيوتر في مؤسساتها وتطوير وحدة المعالجة المركزية في هذه الأجهزة التي


In [24]:
# Apply cleaning to the entire dataset (after we check on sample)
df['text_clean'] = df['text'].apply(clean_arabic_text)

before = len(df)
df = df[df['text_clean'].str.strip() != ''].reset_index(drop=True)
after = len(df)
print(f'Rows before: {before} | Rows after: {after} | Dropped: {before - after}')
df[['text', 'text_clean', 'label']].head(3)

Rows before: 1494 | Rows after: 1494 | Dropped: 0


,text,text_clean,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,human_written
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,human_written
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,human_written


---
## Step 3 – Normalization (Word Variant Handling)


In [25]:
def normalize_arabic(text: str) -> str:
    """Normalize Arabic word variants. Returns a clean string."""
    if not isinstance(text, str):
        return ''

    # Normalize Alef variants → plain Alef
    text = re.sub(r'[أإآٱ]', 'ا', text)

    # Normalize Teh Marbuta → Heh
    text = text.replace('ة', 'ه')

    return text

df['text_normalized'] = df['text_clean'].apply(normalize_arabic)

print('text_normalized dtype:', df['text_normalized'].dtype)
print('Sample (string):', df['text_normalized'].iloc[0][:150])


text_normalized dtype: object
Sample (string): مع زياده الاعتماد على الخدمات الرقميه اصبحت حمايه البيانات الشخصيه لكل منا من اهم التحديات التي تواجهنا حيث تشهد الهجمات الالكترونيه تطورا سريعا في ال


---
## Step 4 – Tokenization


In [26]:
df['tokens_normalized'] = df['text_normalized'].apply(
    lambda t: simple_word_tokenize(t) if isinstance(t, str) else []
)
df['token_count'] = df['tokens_normalized'].apply(len)

print('\nSample tokens (first row):')
print(df['tokens_normalized'].iloc[0][:20])



Sample tokens (first row):
['مع', 'زياده', 'الاعتماد', 'على', 'الخدمات', 'الرقميه', 'اصبحت', 'حمايه', 'البيانات', 'الشخصيه', 'لكل', 'منا', 'من', 'اهم', 'التحديات', 'التي', 'تواجهنا', 'حيث', 'تشهد', 'الهجمات']


---
## Step 5 – Final Dataset Summary & Save

In [27]:
print('=== Final Dataset Overview ===')
print('\nDataFrame columns:')
print(df.columns.tolist())
df[['text', 'text_clean', 'text_normalized', 'tokens_normalized', 'label']].head(5)

=== Final Dataset Overview ===

DataFrame columns:
['text', 'label', 'text_clean', 'text_normalized', 'tokens_normalized', 'token_count']


,text,text_clean,text_normalized,tokens_normalized,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,مع زياده الاعتماد على الخدمات الرقميه اصبحت حم...,"[مع, زياده, الاعتماد, على, الخدمات, الرقميه, ا...",human_written
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,معظم الهجمات التي تحدث وتتم بسرعه وسهوله هي بس...,"[معظم, الهجمات, التي, تحدث, وتتم, بسرعه, وسهول...",human_written
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,"[في, ظل, التطور, التكنولوجي, الكبير, في, هذا, ...",human_written
3,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,يرتبط الامن السيبراني بالذكاء الاصطناعي ارتباط...,"[يرتبط, الامن, السيبراني, بالذكاء, الاصطناعي, ...",human_written
4,الأمن السيبراني هو أحد أهم المجالات في العصر ا...,الأمن السيبراني هو أحد أهم المجالات في العصر ا...,الامن السيبراني هو احد اهم المجالات في العصر ا...,"[الامن, السيبراني, هو, احد, اهم, المجالات, في,...",human_written


In [28]:
TEXT_OUTPUT_FILE = 'text_preprocessed_dataset.csv'
df[['text_normalized', 'label']].to_csv(TEXT_OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved to {TEXT_OUTPUT_FILE}')

TOKENS_OUTPUT_FILE = 'tokens_preprocessed_dataset.csv'
df[['tokens_normalized', 'label']].to_csv(TOKENS_OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved to {TOKENS_OUTPUT_FILE}')


Saved to text_preprocessed_dataset.csv
Saved to tokens_preprocessed_dataset.csv


# - PART 3: Text Representation 


In [84]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

In [85]:
X = df['text_clean']
y = df['label']
print("Number of samples:", len(X))
print(df[['text_clean', 'label']].head())

Number of samples: 1494
                                          text_clean          label
0  مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...  human_written
1  معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...  human_written
2  في ظل التطور التكنولوجي الكبير في هذا العصر وز...  human_written
3  يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...  human_written
4  الأمن السيبراني هو أحد أهم المجالات في العصر ا...  human_written


A) bag of words 

In [86]:
bow_vectorizer = CountVectorizer()

X_bow = bow_vectorizer.fit_transform(X)

# Shape of feature matrix
print("BoW Shape:", X_bow.shape)

print("\nSample Vocabulary:")
print(bow_vectorizer.get_feature_names_out()[:50])

# First document vector
print("\nFirst document vector:")
print(X_bow[0].toarray())



BoW Shape: (1494, 23614)

Sample Vocabulary:
['آب' 'آباء' 'آبائهم' 'آبي' 'آثار' 'آثارا' 'آثاره' 'آثارها' 'آجلا' 'آخذة'
 'آخر' 'آخرها' 'آخرون' 'آخرين' 'آر' 'آراء' 'آرائه' 'آرائهم' 'آرون' 'آسيا'
 'آفات' 'آفاق' 'آفاقا' 'آكل' 'آلات' 'آلاف' 'آلام' 'آلان' 'آلة' 'آله' 'آلي'
 'آليا' 'آليات' 'آلية' 'آليين' 'آمن' 'آمنا' 'آمنة' 'آن' 'آنثروبيك' 'آنفا'
 'آني' 'آنيا' 'آي' 'آية' 'آيفون' 'أباتشي' 'أبحاث' 'أبحاثنا' 'أبدأ']

First document vector:
[[0 0 0 ... 0 0 0]]


In [64]:
bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)
print(bow_df)

# Show first 5 rows and first 20 columns
#display(bow_df.iloc[:5, :20])


      آب  آباء  آبائهم  آبي  آثار  آثارا  آثاره  آثارها  آجلا  آخذة  ...  \
0      0     0       0    0     0      0      0       0     0     0  ...   
1      0     0       0    0     0      0      0       0     0     0  ...   
2      0     0       0    0     0      0      0       0     0     0  ...   
3      0     0       0    0     0      0      0       0     0     0  ...   
4      0     0       0    0     0      0      0       0     0     0  ...   
...   ..   ...     ...  ...   ...    ...    ...     ...   ...   ...  ...   
1489   0     0       0    0     0      0      0       0     0     0  ...   
1490   0     0       0    0     0      0      0       0     0     0  ...   
1491   0     0       0    0     0      0      0       0     0     0  ...   
1492   0     0       0    0     0      0      0       0     0     0  ...   
1493   0     0       0    0     0      0      0       0     0     0  ...   

      يومك  يومنا  يومه  يومهم  يومي  يوميا  يومياته  يومياتهم  يومية  ييا  
0        0

B) TF-IDF

In [74]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

X_tfidf = tfidf_vectorizer.fit_transform(X)

print("TF-IDF Shape:", X_tfidf.shape)

print("\nSample Vocabulary:")
print(tfidf_vectorizer.get_feature_names_out()[:100])

# TF-IDF vector
print(X_tfidf.toarray())

TF-IDF Shape: (1494, 23614)

Sample Vocabulary:
['آب' 'آباء' 'آبائهم' 'آبي' 'آثار' 'آثارا' 'آثاره' 'آثارها' 'آجلا' 'آخذة'
 'آخر' 'آخرها' 'آخرون' 'آخرين' 'آر' 'آراء' 'آرائه' 'آرائهم' 'آرون' 'آسيا'
 'آفات' 'آفاق' 'آفاقا' 'آكل' 'آلات' 'آلاف' 'آلام' 'آلان' 'آلة' 'آله' 'آلي'
 'آليا' 'آليات' 'آلية' 'آليين' 'آمن' 'آمنا' 'آمنة' 'آن' 'آنثروبيك' 'آنفا'
 'آني' 'آنيا' 'آي' 'آية' 'آيفون' 'أباتشي' 'أبحاث' 'أبحاثنا' 'أبدأ' 'أبدا'
 'أبدت' 'أبدته' 'أبراج' 'أبرز' 'أبرزت' 'أبرزها' 'أبرمت' 'أبريل' 'أبسط'
 'أبطأ' 'أبطال' 'أبعاد' 'أبعادا' 'أبعد' 'أبعده' 'أبقى' 'أبل' 'أبلاه'
 'أبلش' 'أبلغ' 'أبناءك' 'أبنائك' 'أبنائهم' 'أبنه' 'أبهرت' 'أبوابا'
 'أبوابها' 'أبوظبي' 'أبون' 'أبونيت' 'أبية' 'أتابع' 'أتاح' 'أتاحت'
 'أتاناسوف' 'أتحدث' 'أتحمم' 'أتخيل' 'أتذكر' 'أترك' 'أتسائل' 'أتصفح'
 'أتطرق' 'أتطلع' 'أتعامل' 'أتعب' 'أتعلم' 'أتعود' 'أتفهم']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [71]:
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print("TF-IDF DataFrame:")
display(tfidf_df.iloc[:5, :20])


first_doc_tfidf = tfidf_df.iloc[0]
non_zero_tfidf = first_doc_tfidf[first_doc_tfidf > 0]
display(non_zero_tfidf.sort_values(ascending=False))

TF-IDF DataFrame:


,آب,آباء,آبائهم,آبي,آثار,آثارا,آثاره,آثارها,آجلا,آخذة,آخر,آخرها,آخرون,آخرين,آر,آراء,آرائه,آرائهم,آرون,آسيا
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


معلوماتنا    0.220807
او           0.202924
بهدف         0.167769
الشخصية      0.162757
يقوم         0.143635
               ...   
قد           0.035755
يمكن         0.034992
في           0.034883
مع           0.029457
هذه          0.024857
Name: 0, Length: 120, dtype: float64

C) WORD2VEC

In [88]:
# Tokenize text
tokenized_texts = [text.split() for text in X]

w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

# Vocabulary size
print("Vocabulary Size:")
print(len(w2v_model.wv))


Vocabulary Size:
11929


In [83]:
# Select one sample document
sample_doc = X.iloc[0]

print("ORIGINAL DOCUMENT")

print(sample_doc)

# Tokenize document
words = sample_doc.split()


print("WORD VECTORS BEFORE AVERAGING")
word_vectors_data = []

for word in words:
    if word in w2v_model.wv:
        vector = w2v_model.wv[word]

  # show first 10 dimensions(each word has 100 dim because vector size 100)
        row = [word] + list(vector[:10])

        word_vectors_data.append(row)

# Create DataFrame
columns = ['word'] + [f'dim_{i}' for i in range(10)]

word_vectors_df = pd.DataFrame(
    word_vectors_data,
    columns=columns
)

display(word_vectors_df)


print("DOCUMENT VECTOR AFTER AVERAGING")
vectors = [
    w2v_model.wv[word]
    for word in words
    if word in w2v_model.wv
]

# Average vectors
avg_vector = np.mean(vectors, axis=0)

# Convert to DataFrame for display
avg_vector_df = pd.DataFrame(
    avg_vector.reshape(1, -1)
)

# Show first 20 dimensions
display(avg_vector_df.iloc[:, :20])

ORIGINAL DOCUMENT
مع زيادة الاعتماد على الخدمات الرقمية أصبحت حماية البيانات الشخصية لكل منا من أهم التحديات التي تواجهنا حيث تشهد الهجمات الإلكترونية تطورا سريعا في السنوات الأخيرة حيث يستخدم المهاجمون تقنيات متقدمة لاختراق الأنظمة وسرقة البيانات وخاصة الحساسة منها الامن السيبراني هو الدرع الأول لحماية الخصوصية والمحافظة على المعلومات الخاصة والحساسة حيث يقوم على مجموعة من الأدوات والسياسات الأمنية وطرق إدارة المخاطر التي يمكن استخدامها في معظم الجهات سواء الحكومية او الخاصة وهو الطريقة التي نحمي بها بياناتها من الهجمات التي يقوم بها بعض المخربين بهدف الحصول على بيانات خاصة والوصول إليها بكل سهولة وبعد ذلك اما يتم استخدامها ضد الجهة التي تم اختراقها او حتى ابتزازها مقابل مبلغ من المال سواء كان ذلك بهدف التخريب او التجريب لذلك يجب علينا الحرص على معلوماتنا الشخصية وعدم اعطائها لأي شخص كان حتى لو كنا نثق به لانه بسبب اهمال بسيط قد تضيع كل هذه المعلومات وهناك عدة طرق للمحافظة على خصوصيتنا وحماية معلوماتنا الشخصية
WORD VECTORS BEFORE AVERAGING


,word,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9
0,مع,0.206325,0.778605,0.728721,0.462127,-0.044555,-1.367502,0.636857,1.533161,-0.606182,-0.545252
1,زيادة,0.114768,0.568292,0.224535,0.217339,-0.089649,-0.790082,0.466903,0.943570,-0.476304,-0.253739
2,الاعتماد,0.027894,0.553445,0.255550,0.324543,-0.256510,-0.584781,0.638058,0.781045,-0.687855,-0.190986
3,على,0.166490,0.695366,-0.113565,0.213224,0.176482,-0.749832,0.921363,1.250223,-0.954963,-0.291391
4,الخدمات,0.113879,0.606215,-0.130136,0.187183,0.038392,-0.842340,0.696856,1.081587,-0.638225,-0.229019
...,...,...,...,...,...,...,...,...,...,...,...
143,على,0.166490,0.695366,-0.113565,0.213224,0.176482,-0.749832,0.921363,1.250223,-0.954963,-0.291391
144,خصوصيتنا,-0.001766,0.025101,0.005999,0.012985,-0.001087,-0.051453,0.034474,0.065281,-0.033415,-0.026230
145,وحماية,0.106203,0.416677,0.131391,0.144521,-0.009032,-0.607462,0.373549,0.750061,-0.366281,-0.203067
146,معلوماتنا,0.013177,0.049455,0.018407,0.023792,-0.005846,-0.082620,0.054092,0.102475,-0.047388,-0.032351


DOCUMENT VECTOR AFTER AVERAGING


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.086475,0.472378,0.089664,0.176571,-0.040107,-0.6797,0.481643,0.817903,-0.498668,-0.230251,-0.07063,-0.620415,-0.259894,0.137648,0.2251,-0.116223,0.283108,-0.261178,-0.178601,-0.913682
